In [0]:
import json
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:

spark.conf.set(
  "fs.azure.account.key.ecommercestorage77.dfs.core.windows.net",
  storage_key
)

In [0]:
bronze_path = "abfss://e-commerce@ecommercestorage77.dfs.core.windows.net/bronze"
silver_path= "abfss://e-commerce@ecommercestorage77.dfs.core.windows.net/Silver"


In [0]:
#Reading raw data from bronze
df_bronze=(
  spark.readStream
  .format("delta")
  .load(bronze_path)
)


In [0]:
#Clean and enrich
df_clean=(
  df_bronze
  .withColumn("timestamp",to_timestamp("timestamp"))
  .withColumn("price",when(col("price").isNull(),0.0).otherwise(col("price")))
  .withColumn("quantity",when(col("quantity").isNull(),1).otherwise(col("quantity")))
  .withColumn("total_amount",col("price")*col("quantity"))
  .dropDuplicates(["order_id"])
  .filter(col("country")=="USA")
  .filter(col("state").isNotNull())
)

In [0]:
#Write to Silver Layer
(
  df_clean.writeStream
 .format("delta")
 .outputMode("append")
 .option("checkpointLocation",silver_path+"/_checkpoint")
 .start(silver_path)
 )

In [0]:
df_silver=spark.read.format("delta").load(silver_path)
display(df_silver)